In [1]:
import mysql.connector
from mysql.connector import Error

In [2]:
from googleapiclient.discovery import build
import os
from dotenv import load_dotenv
load_dotenv()


True

In [3]:
API_KEY = os.getenv("YOUTUBE_API_KEY")
youtube = build('youtube', 'v3', developerKey=API_KEY)
print("YouTube API client created successfully.")

YouTube API client created successfully.


In [ ]:
try:
    connection=mysql.connector.connect(
        host='localhost',
        user="yt_user",
        password=os.getenv("MYSQL_PASSWORD"),
        database='youtube_analytics'
    )
    if connection.is_connected():
        print("Successfully connected to MySQL Platform")
except Error as e:
    print(f"Error connecting to MySQL Platform: {e}")

Successfully connected to MySQL Platform


In [5]:
def insert_channel(connection, channel_data):
    cursor = connection.cursor()

    query = """
    INSERT INTO channels (channel_id, channel_name, subscribers, total_videos, total_views, category)
    VALUES (%s, %s, %s, %s, %s, %s)
    ON DUPLICATE KEY UPDATE
        subscribers = VALUES(subscribers),
        total_videos = VALUES(total_videos),
        total_views = VALUES(total_views),
        category = VALUES(category)
    """

    values = (
        channel_data["channel_id"],
        channel_data["channel_name"],
        channel_data["subscribers"],
        channel_data["total_videos"],
        channel_data["total_views"],
        channel_data["category"]
    )

    cursor.execute(query, values)
    connection.commit()
    cursor.close()


In [6]:
def insert_videos(connection, videos_list):
    cursor = connection.cursor()

    query = """
    INSERT INTO videos (video_id, channel_id, title, published_date, views, likes, comments)
    VALUES (%s, %s, %s, %s, %s, %s, %s)
    ON DUPLICATE KEY UPDATE
        views = VALUES(views),
        likes = VALUES(likes),
        comments = VALUES(comments)
    """

    data = [
        (
            video["video_id"],
            video["channel_id"],
            video["title"],
            video["published_date"],
            video["views"],
            video["likes"],
            video["comments"]
        )
        for video in videos_list
    ]

    cursor.executemany(query, data)
    connection.commit()
    cursor.close()


In [7]:
VIDEO_LIMITS = {
    "big": 1200,
    "medium": 750,
    "small": 400   # None = fetch all
}


In [8]:
def get_video_ids(youtube, channel_id, max_videos=None):
    video_ids = []
    next_page_token = None

    while True:
        request = youtube.search().list(
            part="id",
            channelId=channel_id,
            maxResults=500,
            order="date",
            type="video",
            pageToken=next_page_token
        )

        response = request.execute()

        for item in response["items"]:
            video_ids.append(item["id"]["videoId"])
            if max_videos and len(video_ids) >= max_videos:
                return video_ids[:max_videos]

        next_page_token = response.get("nextPageToken")
        if not next_page_token:
            break

    return video_ids


In [9]:
def fetch_channel_details(youtube, channel_id, category):
    response = youtube.channels().list(
        part="snippet,statistics",
        id=channel_id
    ).execute()

    if not response["items"]:
        return None

    item = response["items"][0]

    return {
        "channel_id": channel_id,
        "channel_name": item["snippet"]["title"],
        "subscribers": int(item["statistics"].get("subscriberCount", 0)),
        "total_videos": int(item["statistics"].get("videoCount", 0)),
        "total_views": int(item["statistics"].get("viewCount", 0)),
        "category": category
    }


In [10]:
def fetch_video_details(youtube, video_ids, channel_id):
    videos = []

    for i in range(0, len(video_ids), 50):
        request = youtube.videos().list(
            part="snippet,statistics",
            id=",".join(video_ids[i:i+50])
        )
        response = request.execute()

        for video in response["items"]:
            videos.append({
                "video_id": video["id"],
                "channel_id": channel_id,
                "title": video["snippet"]["title"],
                "published_date": video["snippet"]["publishedAt"].replace("T", " ").replace("Z", ""),
                "views": int(video["statistics"].get("viewCount", 0)),
                "likes": int(video["statistics"].get("likeCount", 0)),
                "comments": int(video["statistics"].get("commentCount", 0))
            })

    return videos


In [11]:
try:
    import pandas as pd

    channels_df = pd.read_csv("../data/channel_ids.csv")

    for idx, row in channels_df.iterrows():
        channel_id = row["channel_id"]
        category = row["category"]
        print(f"Processing {channel_id} ({category})")
        channel_data = fetch_channel_details(youtube, channel_id, category)
        if not channel_data:
            continue

        insert_channel(connection, channel_data)

        limit = VIDEO_LIMITS[category]
        video_ids = get_video_ids(youtube, channel_id, limit)

        if video_ids:
            videos_data = fetch_video_details(youtube, video_ids, channel_id)
            insert_videos(connection, videos_data)

        print(f"Completed {channel_id}")
        

except Exception as e:
    print("Error:", e)


Processing UCCpSTBr5nxLCyMEsyTSxNEA (big)
Completed UCCpSTBr5nxLCyMEsyTSxNEA
Processing UCH_7bF90ySEVfWLXtn9HlWA (medium)
Completed UCH_7bF90ySEVfWLXtn9HlWA
Processing UCto7D1L-MiRoOziCXK9uT5Q (big)
Completed UCto7D1L-MiRoOziCXK9uT5Q
Processing UC0jPwNs4B7yJySNewHan5hQ (big)
Completed UC0jPwNs4B7yJySNewHan5hQ
Processing UCiV37YIYars6msmIQXopIeQ (small)
Completed UCiV37YIYars6msmIQXopIeQ
Processing UC41_XBfZbfRHrNB9c9pD1OA (medium)
Completed UC41_XBfZbfRHrNB9c9pD1OA
Processing UCXUJJNoP1QupwsYIWFXmsZg (big)
Completed UCXUJJNoP1QupwsYIWFXmsZg
Processing UCiNu3bxmSb7RRkYfIilnDPQ (small)
Completed UCiNu3bxmSb7RRkYfIilnDPQ
Processing UC7REFmsnHu34TNLxLCGiEcQ (small)
Completed UC7REFmsnHu34TNLxLCGiEcQ
Processing UC3xKgYs1GqtoPu9tyYSjR8w (small)
Completed UC3xKgYs1GqtoPu9tyYSjR8w
Processing UCA6q5E9yd7cSgl8m0TrxOmQ (big)
Completed UCA6q5E9yd7cSgl8m0TrxOmQ
Processing UCcnce86Js0V-D5k9-0XUSGA (small)
Completed UCcnce86Js0V-D5k9-0XUSGA
Processing UCdp4_l1vPmpN-gDbUwhaRUQ (big)
Completed UCdp4_l1